# Event-Specific DInSAR Deformation Mapping
### Case study: 2016 Amatrice–Norcia earthquake sequence (Central Italy)

This notebook implements **use case 1**: a *single-epoch / few-pair* DInSAR deformation
snapshot, built entirely from openEO User-Defined Processes (UDPs) running on the
Copernicus Data Space Ecosystem (CDSE):

- [`sentinel1_sar_interferogram`](https://algorithm-catalogue.apex.esa.int/apps/sentinel1_sar_interferogram) — geocoded wrapped/unwrapped interferogram + coherence
- [`sentinel1_sar_coherence`](https://algorithm-catalogue.apex.esa.int/apps/sentinel1_sar_coherence) — geocoded coherence (used here only to help pick a good pair)

Unlike SBAS/PSI time-series inversion, this workflow only needs **one or two independent
interferometric pairs** bracketing a discrete deformation event (a moderate earthquake),
so it stays cheap enough to run interactively.

**Area of interest:** Amatrice, Central Italy — epicentral area of the Mw 6.2 earthquake
of 24 August 2016 (Amatrice–Norcia seismic sequence), which produced a well-documented
co-seismic surface deformation pattern clearly visible in Sentinel-1 InSAR (e.g.
Scognamiglio et al., 2016; Chiaraluce et al., 2017; Walters et al., 2018). The AOI sits
entirely within Europe.

**What this notebook does:**
1. Discovers a Sentinel-1 burst covering the AOI (openEO's InSAR UDPs operate at burst level).
2. Lists available Sentinel-1 SLC acquisition dates around the earthquake so you can pick a
   real pre-/post-event pair (hardcoded dates would likely not match actual acquisitions).
3. Runs `sentinel1_sar_interferogram` for one (or more) independent pairs.
4. Applies a **custom UDF** that converts unwrapped phase → line-of-sight (LOS) displacement,
   masks low-coherence pixels, flags residual-fringe / unwrapping-error artifacts, and (if more
   than one independent pair is available) does a simple robust stack — **not** a network
   inversion.
5. Visualises the resulting co-seismic LOS displacement map.

> This is a template. Sentinel-1's exact acquisition dates/orbits over Amatrice on the
> dates around the earthquake, and the exact burst_id, must be confirmed from the catalogue
> (step 2 below) before running the job — adjust the placeholders as needed.


In [1]:
# --- Imports ---
import json
import requests
import numpy as np
import xarray as xr
import matplotlib.pyplot as plt

import openeo


In [2]:
# --- Connect to the openEO back-end on CDSE ---
connection = openeo.connect("openeo.dataspace.copernicus.eu").authenticate_oidc()


Authenticated using refresh token.


## 1. Area of interest

A small box around Amatrice, Italy (epicentre of the 24 Aug 2016 Mw 6.2 event).

In [3]:
aoi = {
    "type": "Polygon",
    "coordinates": [
        [
            [13.10, 42.60],
            [13.10, 42.78],
            [13.38, 42.78],
            [13.38, 42.60],
            [13.10, 42.60],
        ]
    ],
}


## 2. Find a Sentinel-1 burst covering the AOI

`sentinel1_sar_interferogram` / `sentinel1_sar_coherence` operate on individual Sentinel-1
TOPS **bursts** (not full scenes), so we need a `burst_id` and `sub_swath` for our AOI.
We query the CDSE OData `Bursts` endpoint, filtering by a bounding-box intersection and a
date range around the event, and inspect the candidates.

Docs: https://documentation.dataspace.copernicus.eu/APIs/Sentinel-1%20SLC%20Burst.html


In [4]:
def find_candidate_bursts(aoi_polygon, start, end, top=20):
    coords = aoi_polygon["coordinates"][0]
    wkt_coords = ", ".join(f"{lon} {lat}" for lon, lat in coords)
    footprint_wkt = f"POLYGON(({wkt_coords}))"

    filter_str = (
        f"OData.CSC.Intersects(area=geography'SRID=4326;{footprint_wkt}') "
        f"and ContentDate/Start gt {start}T00:00:00.000Z "
        f"and ContentDate/Start lt {end}T00:00:00.000Z "
        f"and PolarisationChannels eq 'VV'"
    )
    url = (
        "https://catalogue.dataspace.copernicus.eu/odata/v1/Bursts"
        f"?$filter={filter_str}&$top={top}&$orderby=ContentDate/Start asc"
    )
    resp = requests.get(url)
    resp.raise_for_status()
    return resp.json().get("value", [])


# Search a generous window around the mainshock so we have pre- and post-event candidates
candidates = find_candidate_bursts(aoi, "2016-08-01", "2016-09-15")

for b in candidates:
    print(
        b.get("Id"), "|",
        "burst_id:", b.get("BurstId"),
        "swath:", b.get("SwathIdentifier"),
        "orbit:", b.get("RelativeOrbitNumber"),
        "direction:", b.get("OrbitDirection"),
        "date:", b.get("ContentDate", {}).get("Start"),
    )


a21db0af-2503-44f1-ae30-95ef4c5b6580 | burst_id: 202725 swath: IW1 orbit: 95 direction: DESCENDING date: 2016-08-02T05:19:40.757120Z
da361800-ca1e-4e99-af21-bfb96efdcc99 | burst_id: 202725 swath: IW1 orbit: 95 direction: DESCENDING date: 2016-08-02T05:19:40.757120Z
33bc5ec4-b3cb-46eb-8383-25b762cde40d | burst_id: 202726 swath: IW1 orbit: 95 direction: DESCENDING date: 2016-08-02T05:19:43.515397Z
ed5263c8-744a-495b-90de-1a33d8f524c3 | burst_id: 202726 swath: IW1 orbit: 95 direction: DESCENDING date: 2016-08-02T05:19:43.515397Z
76bc77da-213c-4a79-96e3-9c4951a59988 | burst_id: 249409 swath: IW3 orbit: 117 direction: ASCENDING date: 2016-08-03T17:05:50.192434Z
a5315037-f5de-465e-9bfb-31f74092de64 | burst_id: 249410 swath: IW3 orbit: 117 direction: ASCENDING date: 2016-08-03T17:05:52.950710Z
c043912e-2881-4d91-9bb0-f0f613b371eb | burst_id: 45927 swath: IW2 orbit: 22 direction: DESCENDING date: 2016-08-09T05:11:30.081838Z
228ade0e-e970-4117-b871-2fc12e9e708a | burst_id: 45927 swath: IW3 orbi

> **Note:** the Sentinel-1 SLC Burst catalogue on CDSE only indexes bursts consistently
> from August 2024 onward for direct OData querying; for older acquisitions (like this 2016
> event) you may need to cross-check via the general `SENTINEL-1` OData/STAC catalogue
> (`Collection eq 'SENTINEL-1'`, `ProductType eq 'IW_SLC__1S'`) to confirm exact acquisition
> dates and relative orbit, and then supply the corresponding `burst_id`/`sub_swath` manually
> below. If you already know the track/orbit that covers Amatrice, tell me and I can hardcode it.

## 3. Fix the processing parameters

Fill these in from the discovery step above. The values below are **placeholders** —
replace `BURST_ID` / `SUB_SWATH` with a real candidate, and `PRE_EVENT_DATE` /
`POST_EVENT_DATE` with two real, independent acquisition dates that bracket
2016-08-24 (mainshock) — and optionally a third date bracketing 2016-10-30
(Mw 6.5 Norcia event) if you want a second, independent snapshot.


In [5]:
BURST_ID = 202725           # <-- replace with a real burst_id for the AOI
SUB_SWATH = "IW1"          # <-- IW1 / IW2 / IW3, replace as needed
POLARIZATION = "VV"

# Independent interferometric pairs bracketing the event(s) - NOT a redundant SBAS network
INSAR_PAIRS = [
    ["2016-08-18", "2016-08-24"],   # brackets the 24 Aug Mw 6.2 mainshock
    # ["2016-10-24", "2016-10-30"],  # optional: brackets the 30 Oct Mw 6.5 Norcia event
]


## 4. Run `sentinel1_sar_interferogram`

Each entry in `InSAR_pairs` is processed as an **independent interferogram** — there is no
network inversion across pairs.

In [6]:
s1_interferogram = connection.datacube_from_process(
    "sentinel1_sar_interferogram",
    namespace=(
        "https://raw.githubusercontent.com/ESA-APEx/apex_algorithms/refs/heads/main/"
        "algorithm_catalog/eurac/sentinel1_sar_interferogram/openeo_udp/"
        "sentinel1_sar_interferogram.json"
    ),
    **{
        "InSAR_pairs": INSAR_PAIRS,
        "burst_id": BURST_ID,
        "coherence_window_az": 2,
        "coherence_window_rg": 10,
        "n_az_looks": 1,
        "n_rg_looks": 4,
        "polarization": POLARIZATION,
        "sub_swath": SUB_SWATH,
    },
)


It's worth inspecting the resulting band names before writing the UDF, since the exact
labels (e.g. `coherence`, `phase_unwrapped`, `phase_wrapped`) depend on the UDP version:

```python
print(s1_interferogram.metadata)
```

The UDF below assumes bands named `"coherence"` and `"phase_unwrapped"` — adjust the
`.sel(bands=...)` calls if your job's actual band labels differ.

## 5. UDF: unwrapped phase → LOS displacement, with masking & artifact flagging

This is the part that replaces a full geodetic inversion: for a **single pair**, converting
phase to displacement is a direct, closed-form calculation per pixel — no redundant network,
no least-squares solve.

`disp = -(λ / 4π) · Δφ` where λ = 0.05546 m is the Sentinel-1 C-band wavelength.


In [7]:
phase_to_displacement_udf = """
import numpy as np
import xarray as xr

LAMBDA_M = 0.05546          # Sentinel-1 C-band wavelength [m]
COHERENCE_THRESHOLD = 0.30  # mask pixels below this coherence
GRADIENT_PERCENTILE = 98    # flag steepest local-phase-gradient pixels as artifacts


def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
    coherence = cube.sel(bands="coherence")
    phase = cube.sel(bands="phase_unwrapped")

    # 1) convert phase to LOS displacement
    displacement = -(LAMBDA_M / (4 * np.pi)) * phase

    # 2) coherence mask - discard unreliable / decorrelated pixels
    displacement = displacement.where(coherence >= COHERENCE_THRESHOLD)

    # 3) artifact flag - large local phase gradients usually indicate a residual
    #    fringe or a local unwrapping error rather than real ground motion
    dy = phase.differentiate("y")
    dx = phase.differentiate("x")
    gradient_mag = np.sqrt(dx ** 2 + dy ** 2)
    gradient_cutoff = gradient_mag.quantile(GRADIENT_PERCENTILE / 100.0)
    displacement = displacement.where(gradient_mag < gradient_cutoff)

    result = xr.concat([displacement, coherence], dim="bands")
    result = result.assign_coords(bands=["los_displacement_m", "coherence"])
    return result
"""


In [11]:
s1_displacement = s1_interferogram.apply_dimension(
    code=phase_to_displacement_udf,
)


C:\Users\SHARMAP\AppData\Local\Temp\ipykernel_29416\1113869058.py:1: UserDeprecationWarning: Specifying UDF code through `code`, `runtime` and `version` arguments is deprecated. Instead create an `openeo.UDF` object and pass that to the `process` argument.
  s1_displacement = s1_interferogram.apply_dimension(
C:\Users\SHARMAP\AppData\Local\Temp\ipykernel_29416\1113869058.py:1: UserDeprecationWarning: Using `DataCube.apply_dimension()` without explicit `dimension` argument is deprecated and will become invalid in a future version. Falling back on dimension 't'.
  s1_displacement = s1_interferogram.apply_dimension(


## 6. (Optional) robust stack across independent pairs

If you supplied more than one independent pair in `INSAR_PAIRS` (e.g. one per seismic
event, or repeated acquisitions of the same event for noise reduction), take a coherence-
weighted median across the pair dimension. This is basic stacking for noise reduction,
**not** an SBAS/PSI least-squares network inversion.

In [12]:
# If InSAR_pairs has more than one entry, the 't' (or 'pair') dimension holds each
# independent result. A simple robust combination:
#
# stacked_udf = """
# import numpy as np
# import xarray as xr
#
# def apply_datacube(cube: xr.DataArray, context: dict) -> xr.DataArray:
#     displacement = cube.sel(bands="los_displacement_m")
#     return displacement.median(dim="t", skipna=True)
# """
#
# s1_displacement_stacked = s1_displacement.apply_dimension(
#     code=stacked_udf, runtime="Python", dimension="t"
# )


## 7. Execute the batch job

In [13]:
job = s1_displacement.create_job(
    title="amatrice_coseismic_displacement",
    outputformat="netCDF",
)
job.start_and_wait()
# job.get_results().download_files("amatrice_displacement")


Preflight process graph validation raised: [ProcessParameterInvalid] The value passed for parameter 'data' in process 'apply_dimension' is invalid: Expected raster cube or vector cube but got <class 'openeogeotrellis.stac_save_result.StacSaveResult'>.


0:00:00 Job 'j-26072809070247aca2401f94fc02eb6a': send 'start'


OpenEoApiError: [400] ProcessParameterInvalid: The value passed for parameter 'data' in process 'apply_dimension' is invalid: Expected raster cube or vector cube but got <class 'openeogeotrellis.stac_save_result.StacSaveResult'>. (ref: r-2607280907054feeb68b911f425cf90d)

## 8. Plot the co-seismic LOS displacement map

In [ ]:
result_ds = xr.load_dataset("amatrice_displacement/openEO.nc")

displacement = result_ds["los_displacement_m"] if "los_displacement_m" in result_ds else (
    result_ds.to_array(dim="bands").sel(bands="los_displacement_m")
)

fig, ax = plt.subplots(figsize=(6, 6), dpi=100)
im = displacement.squeeze().plot.imshow(ax=ax, cmap="RdBu_r", vmin=-0.15, vmax=0.15,
                                         add_colorbar=True, cbar_kwargs={"label": "LOS displacement [m]"})
ax.set_title("Co-seismic LOS displacement — Amatrice, 24 Aug 2016")
ax.set_xlabel("")
ax.set_ylabel("")
plt.tight_layout()
plt.show()


## Notes & limitations

- This gives a **single-epoch LOS displacement snapshot**, not a deformation *rate* or
  time series — exactly the right scope for a discrete event like an earthquake, and far
  cheaper than SBAS/PSI.
- Positive/negative sign convention depends on ascending vs. descending geometry and needs
  to be checked against the orbit direction of the pair used.
- Atmospheric phase screen (APS) is **not** corrected here; for a single pair this is a
  known limitation — cross-checking against a second, independent pair (different track or
  date) is a cheap sanity check without needing a full inversion.
- If later you want deformation *evolution* (post-seismic transient, afterslip), that's
  exactly the case where a real SBAS/PSI time-series inversion becomes worth the extra cost.
